# **Naive RAG**

Naive RAG is a basic Retrieval-Augmented Generation approach that retrieves relevant information from a knowledge source and provides it to an LLM to generate a context-aware response.

**Install libraries**

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-groq \
    pypdf

**Get the APIKEY**

In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully!")

**PDF upload**

In [ ]:
from google.colab import files

uploaded = files.upload()

**Load PDF**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

**Chunking**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunking completed!")
print("Number of chunks:", len(chunks))

**Embedding**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Embeddings created!")

**Vectore database**

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database created!")

**Create Retriever**

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

**Ask Question**

In [ ]:
question = input("Ask: ")

**Retrieve Relevant Information**

In [ ]:
docs = retriever.invoke(question)

for doc in docs:
    print(doc.page_content)

**Connect Groq**

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    api_key=GROQ_API_KEY
)

**Give Context to LLM**

In [16]:
context = "\n\n".join(
    doc.page_content for doc in docs
)

prompt = f"""
Answer using only this context:

{context}

Question:
{question}
"""

**Generate Answer**

In [ ]:
answer = llm.invoke(prompt)

print(answer.content)